# 05 baseline

Step 5 of the guide: the opponent. Wells Fargo compared their transformer with an XGBoost on about 100 hand-built features; here it is XGBoost on 29 features in four groups, built with care so that a transformer win means something. The same rows, the same split and the same metrics as step 4.

```
raw row i  ->  current (6)    amount, category code, hour, weekday, distance
           ->  gaps (3)       seconds since the card's previous row / same-category row / same-merchant row
           ->  rolling (8)    count and amount over the last 1 h, 24 h, 7 d, 30 d of the same card, window [t - w, t)
           ->  deviation (12) row i against the card's whole past: mean, std, z-score, category / merchant / hour share, first visit
```

| setting | value | guide |
|---|---|---|
| features | 29 (WF: about 100) | 25 to 40, designed with care |
| history windows | `rolling(w, on="ts", closed="left")`: rows strictly before the current one | including the row itself leaks |
| split | the step-1 `split` column, identical to the transformer's | same train / val / test |
| selection | early stopping on validation PR-AUC; grid over max_depth, learning_rate, scale_pos_weight on validation | the three knobs the guide names |
| control | the same XGBoost on the 6 current-transaction features only | "how much does history matter" |
| seeds | 0, 1, 2 per variant (subsample and colsample make them differ) | mean and sd |
| cost | seconds to compute the features, and the time it took to write them | WF's selling point is "no feature engineering" |

In [ ]:
import os, sys, glob, shutil
try:
    from google.colab import drive, userdata                      # Colab: mount Drive, pull the repo, sync new notebooks to Drive
    drive.mount("/content/drive")
    tok = userdata.get("GH_TOKEN")
    if not os.path.exists("/content/transaction-fm"):
        !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
    else:
        !cd /content/transaction-fm && git pull -q
    %pip install -q datasets xgboost
    %cd /content/transaction-fm
except ImportError:                                               # laptop: this notebook is the repo copy; data under $TFM_ROOT
    here = os.getcwd()
    while not os.path.isdir(os.path.join(here, "src", "tfm")):
        assert here != os.path.dirname(here), "run this notebook from inside the transaction-fm repo"
        here = os.path.dirname(here)
    %cd {here}
sys.path.insert(0, os.path.abspath("src"))
from tfm.runtime import root, describe, on_colab
ROOT = root()
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
if on_colab():
    for f in glob.glob("notebooks/*.ipynb"):                      # new notebooks from the repo appear in Drive
        dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok |", describe(), "| ROOT =", ROOT)

## What trains on what

- Every history feature of row i is computed from rows strictly before i on the same card (`closed="left"`, `cumsum - current`), so a row never sees itself or anything later. `tests/test_baseline.py` edits a later row and checks earlier rows' features are unchanged, and edits the current row's amount and checks its pure-history features are unchanged.
- The only fitted encoder is category -> integer code, fit on train rows through `assert_train_only`; an unseen category becomes -1.
- `is_fraud` and `split` never enter a feature; the test flips every label and checks the feature matrix is identical.
- XGBoost is fit on train rows only; validation chooses the number of trees and the grid point; test rows are scored once per run.
- Cardholder age from `cards.parquet` is deliberately not a feature although the guide lists it: the transformer does not see it either, and step 1 reserved that table for step-6 subgroup checks only. A fair baseline sees the same columns as the model it is compared with.

## Stage 5a: build the 29 features

In [ ]:
import json, time, os, glob, numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
from tfm.baseline import FEATURES, CURRENT_ONLY, GROUPS, DESCRIPTION, build_features
from tfm.metrics import evaluate_scores, mean_sd

P, RES = f"{ROOT}/data/processed", f"{ROOT}/results"
df = pd.read_parquet(f"{P}/transactions.parquet")
t0 = time.time(); feats = build_features(df); FEAT_SECONDS = round(time.time() - t0, 1)
print(f"{len(FEATURES)} features for {len(df):,} rows in {FEAT_SECONDS} s; {feats.memory_usage().sum() / 1e6:.0f} MB")

card, split, fraud = df["card_id"].to_numpy(), df["split"].to_numpy(), df["is_fraud"].to_numpy()
tr, va, te = split == "train", split == "val", split == "test"
table = pd.DataFrame([{"group": g, "feature": c, "what": DESCRIPTION[c], "dtype": str(feats[c].dtype),
                       "NaN share (train)": float(feats.loc[tr, c].isna().mean()), "median (train)": float(feats.loc[tr, c].median())}
                      for g, cols in GROUPS.items() for c in cols])
display(table.style.format({"NaN share (train)": "{:.1%}", "median (train)": "{:,.2f}"}).hide(axis="index"))
print("NaN means 'no earlier row to compute this from' (a card's first rows); XGBoost treats it as its own branch")

## Stage 5b: one card's rows next to their features, then a machine check

The first fraud in the test period with the five rows before it: raw columns (with `is_fraud`, for the reader only) and a few of the history features. Reading down `n_24h` and `hist_n` shows what "strictly before" means: the row's own amount is not in `amt_24h`, and `hist_mean_amt` is the mean of the rows above it. Then 2,000 random rows are recomputed by brute force from the same card's earlier rows and compared with the feature frame.

In [ ]:
i = int(np.flatnonzero(te & (fraud == 1))[0]); c0 = card[i]
lo = max(int(np.flatnonzero(card == c0)[0]), i - 5)
show = pd.concat([df.iloc[lo: i + 1][["card_id", "split", "ts", "category", "merchant", "amt", "is_fraud"]],
                  feats.iloc[lo: i + 1][["gap_prev_s", "n_24h", "amt_24h", "hist_n", "hist_mean_amt", "amt_ratio_mean", "hist_mer_n", "first_at_merchant"]]], axis=1)
display(show.style.format({"amt": "{:.2f}", "amt_24h": "{:.2f}", "hist_mean_amt": "{:.2f}", "amt_ratio_mean": "{:.2f}", "gap_prev_s": "{:,.0f}"}))

rng = np.random.default_rng(0); bad = 0; checked = 0
ts = df["ts"].to_numpy(); amt = df["amt"].to_numpy(); mer = df["merchant"].to_numpy()
starts = np.flatnonzero(np.r_[True, card[1:] != card[:-1]]); card_start = np.repeat(starts, np.diff(np.r_[starts, len(card)]))
for k in rng.choice(len(df), min(2000, len(df)), replace=False):
    s = card_start[k]; past = slice(s, k)                                   # same card, rows strictly before k
    in24 = (ts[past] >= ts[k] - np.timedelta64(24, "h")) & (ts[past] < ts[k])
    ok = feats.at[k, "n_24h"] == in24.sum() and np.isclose(feats.at[k, "amt_24h"], amt[past][in24].sum())
    ok &= feats.at[k, "hist_n"] == k - s and feats.at[k, "hist_mer_n"] == (mer[past] == mer[k]).sum()
    ok &= (k == s and np.isnan(feats.at[k, "hist_mean_amt"])) or (k > s and np.isclose(feats.at[k, "hist_mean_amt"], amt[past].mean()))
    ok &= (k == s and np.isnan(feats.at[k, "gap_prev_s"])) or (k > s and feats.at[k, "gap_prev_s"] == (ts[k] - ts[k - 1]) / np.timedelta64(1, "s"))
    bad += not ok; checked += 1
print(f"machine check on {checked:,} random rows: {checked - bad:,} consistent, {bad} inconsistent")
assert bad == 0

## Stage 5c: the grid, selected on validation

12 fits (max_depth 4 / 6 / 8, learning_rate 0.05 / 0.1, scale_pos_weight 1 / 20), seed 0, up to 2,000 trees each with early stopping after 50 rounds without a validation PR-AUC gain. The table is saved, so a rerun of this cell after a disconnect reads it instead of refitting.

In [ ]:
from tfm.baseline import grid_search, fit_xgb
from tfm.runtime import xgb_device
DEV = xgb_device()
print("xgboost device:", DEV, describe(), "" if DEV == "cuda" else "(no CUDA: hist on all CPU cores)")
Xtr, ytr, Xv, yv, Xt, yt = feats[tr], fraud[tr], feats[va], fraud[va], feats[te], fraud[te]
GRID = [dict(max_depth=d, learning_rate=lr, scale_pos_weight=w) for d in (4, 6, 8) for lr in (0.05, 0.1) for w in (1.0, 20.0)]
gpath = f"{RES}/05_grid.json"
if os.path.exists(gpath):
    grid = pd.read_json(gpath); print("grid read from", gpath)
else:
    grid = grid_search(Xtr, ytr, Xv, yv, GRID, seed=0, device=DEV); grid.to_json(gpath, orient="records", indent=2)
display(grid.style.format({"val_pr_auc": "{:.4f}", "learning_rate": "{:.2f}"}).hide(axis="index"))
BEST = {k: grid.iloc[0][k] for k in ("max_depth", "learning_rate", "scale_pos_weight")}; BEST["max_depth"] = int(BEST["max_depth"])
print("best on validation:", BEST, f"({int(grid.iloc[0]['best_iteration']) + 1} trees)")
!PYTHONPATH=src python -m pytest -q tests 2>&1 | tail -1

## Stage 5d: six fits (2 variants x 3 seeds), each with its own result file

`full` uses the 29 features, `current` the 6 current-transaction ones, both with the config chosen above. A fit whose json exists is skipped. Test scores are saved as `test_scores.npy` under `baseline/<variant>/seed<k>/` for step 6 (bootstrap, error analysis), and the seed-0 full model's feature importance (gain) goes into its json.

In [ ]:
RUNS = [(variant, seed) for variant in ("full", "current") for seed in (0, 1, 2)]
COLS = {"full": FEATURES, "current": CURRENT_ONLY}
for variant, seed in RUNS:
    name = f"05_xgb_{variant}_seed{seed}"; out = f"{ROOT}/baseline/{variant}/seed{seed}"; res = f"{RES}/{name}.json"
    if os.path.exists(res):
        print(f"{name}: done earlier, skipped"); continue
    cols = COLS[variant]
    model, info = fit_xgb(Xtr[cols], ytr, Xv[cols], yv, seed=seed, device=DEV, **BEST)
    p_test = model.predict_proba(Xt[cols])[:, 1]
    os.makedirs(out, exist_ok=True); np.save(f"{out}/test_scores.npy", p_test.astype(np.float32))
    test_m = evaluate_scores(yt, p_test, card[te], 0.006)
    gain = model.get_booster().get_score(importance_type="gain")
    summary = {"name": name, "variant": variant, "n_features": len(cols), "seed": seed, **info, "n_train_rows": int(tr.sum()), "n_train_fraud": int(ytr.sum()),
               "test": test_m, "feature_seconds": FEAT_SECONDS, "device": DEV, "runtime": describe(), "importance_gain": {k: round(v, 2) for k, v in sorted(gain.items(), key=lambda kv: -kv[1])}}
    with open(res, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"{name}: {info['best_iteration'] + 1} trees  val PR-AUC {info['val_pr_auc']:.4f}  |  test PR-AUC {test_m['pr_auc']:.4f}  "
          f"recall@0.6% {test_m['recall_at_alert']:.3f}  card discovery {test_m['card_discovery']:.3f}  FP ratio {test_m['fp_ratio']:.2f}  |  {info['fit_seconds']} s")

## Stage 5e: results, mean and sd, against the transformer

One row per fit, the aggregate per variant, the history gain (full minus current-only), then the comparison the whole project is about: XGBoost full vs the step-4 pretrained transformer vs the scratch transformer, all on the same test rows. Last, the ten features XGBoost leaned on most: a validator reads this next to the step-6 history ablation.

In [ ]:
runs = [json.load(open(f)) for f in sorted(glob.glob(f"{RES}/05_xgb_*_seed*.json"))]
KEYS = ["pr_auc", "roc_auc", "recall_at_alert", "card_discovery", "fp_ratio"]
flat = pd.DataFrame([{"variant": r["variant"], "seed": r["seed"], "trees": r["best_iteration"] + 1, "val PR-AUC": r["val_pr_auc"],
                      **{f"test {k}": r["test"][k] for k in KEYS}, "fit s": r["fit_seconds"]} for r in runs])
display(flat.style.format(precision=4).hide(axis="index"))
agg = {v: mean_sd([r["test"] for r in runs if r["variant"] == v]) for v in ("full", "current") if any(r["variant"] == v for r in runs)}
history_gain = {}
if {"full", "current"} <= set(agg):
    for k in KEYS:
        history_gain[k] = agg["full"][k]["mean"] - agg["current"][k]["mean"]
        print(f"{k:16s} full {agg['full'][k]['mean']:.4f} +- {agg['full'][k]['sd']:.4f}   current-only {agg['current'][k]['mean']:.4f} +- {agg['current'][k]['sd']:.4f}   history gain {history_gain[k]:+.4f}")

ft_path = next(p for p in (f"{RES}/04_finetune.json", "results/metrics/04_finetune.json") if os.path.exists(p))   # Drive copy, else the repo copy
ft = json.load(open(ft_path))["aggregate_test"]
rows = {"XGBoost 29 features": agg.get("full"), "XGBoost current only": agg.get("current"), "transformer pretrained": ft.get("pretrained"), "transformer scratch": ft.get("scratch")}
cmp = pd.DataFrame({name: {k: f"{a[k]['mean']:.4f} +- {a[k]['sd']:.4f}" for k in KEYS} for name, a in rows.items() if a})
display(cmp)
if agg.get("full") and ft.get("pretrained"):
    for k in KEYS:
        print(f"{k:16s} transformer minus XGBoost {ft['pretrained'][k]['mean'] - agg['full'][k]['mean']:+.4f}")

imp = pd.Series(next(r for r in runs if r["variant"] == "full" and r["seed"] == 0)["importance_gain"]).head(10)
display(imp.rename("gain, seed 0 full model").to_frame().style.format("{:,.1f}"))
print(f"\nfeature cost: {len(FEATURES)} features computed in {runs[0]['feature_seconds']} s for {len(df):,} rows; authoring time is in the README (commit log)")

## Stage 5f: precision-recall curves with the transformer, and the step-5 json

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
fig, ax = plt.subplots(figsize=(6, 4.5))
curves = {"XGBoost 29 features": f"{ROOT}/baseline/full/seed0/test_scores.npy", "XGBoost current only": f"{ROOT}/baseline/current/seed0/test_scores.npy",
          "transformer pretrained": f"{ROOT}/finetune/pretrained_H12/seed0/test_scores.npy"}
for label, path in curves.items():
    if not os.path.exists(path): continue
    p = np.load(path); assert len(p) == len(yt)                     # both are the test rows in parquet order
    from sklearn.metrics import average_precision_score
    prec, rec, _ = precision_recall_curve(yt, p); ax.plot(rec, prec, label=f"{label} ({average_precision_score(yt, p):.3f})")
ax.axhline(float(yt.mean()), ls="--", c="gray", label=f"fraud rate {yt.mean():.2%}")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title("test set, seed 0"); ax.legend(); plt.tight_layout()
fig.savefig(f"{RES}/05_baseline_pr.png", dpi=120); plt.show()

out = {"n_features": len(FEATURES), "features": GROUPS, "best_config": BEST, "grid": grid.to_dict(orient="records"), "feature_seconds": FEAT_SECONDS,
       "n_test": int(len(yt)), "n_test_fraud": int(yt.sum()), "runs": runs, "aggregate_test": agg, "history_gain": history_gain,
       "transformer_minus_xgb": {k: ft["pretrained"][k]["mean"] - agg["full"][k]["mean"] for k in KEYS} if agg.get("full") and ft.get("pretrained") else {}}
with open(f"{RES}/05_baseline.json", "w") as f:
    json.dump(out, f, indent=2)
print("saved 05_baseline.json:", list(out))

## What to read off the numbers

- `transformer minus XGBoost` on PR-AUC and recall at 0.6% alerts is the project's headline. Either sign is a result: the guide says a well-argued XGBoost win on small data is as good a finding as a transformer win. Step 6 puts a bootstrap confidence interval on it.
- `history gain` (full minus current-only) says how much the card's past is worth to a tree model. Compare it with the step-6 history-length ablation (0 / 5 / 12 / 32) on the transformer: the two should tell the same story.
- The importance table is the validator's cross-check: if `gap_prev_s`, `hist_hour_share` and `amt_ratio_mean` dominate, the model is reading timing and deviation from the card's habits, which is what the transformer must learn from tokens on its own.
- Scores near 1.0 for every model are the Sparkov caveat again: rule-generated fraud. The fixed-alert-rate gaps and the ablation trends carry the report, not the absolute level.

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
if on_colab():
    def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
        cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
        return cells(a) == cells(b)
    for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
        repo = f"notebooks/{os.path.basename(f)}"
        if not os.path.exists(repo) or same_code(f, repo):
            shutil.copy(f, repo)
        else:
            print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
    !git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
    !git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
    !git log --oneline | head -1
else:
    print("laptop: results copied into results/; this notebook file already is the repo copy; commit from the terminal")
    !git status --short results notebooks

In [ ]:
if on_colab():
    drive.flush_and_unmount()